In [7]:
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# Core helpers
# =========================================================
def project_to_domain(x, domain):
    return min(max(x, domain[0]), domain[1])

def eval_loss_and_grad(loss_t, x):
    if loss_t["type"] == "linear":
        g = loss_t["g"]
        return g * x, g
    elif loss_t["type"] == "quadratic":
        q = loss_t["q"]; c = loss_t["c"]
        return 0.5*q*x*x - q*c*x, q*(x - c)
    else:
        raise ValueError("Unknown loss type")

def update_aggregates(A, B, loss_t):
    if loss_t["type"] == "linear":
        return A, B + loss_t["g"]
    else:
        return A + loss_t["q"], B - loss_t["q"] * loss_t["c"]

def argmin_over_domain(A, B, domain):
    if A > 0:
        x_star = -B / A
    else:
        if B > 0: x_star = domain[0]
        elif B < 0: x_star = domain[1]
        else: x_star = 0.0
    return project_to_domain(x_star, domain)

def hindsight_min_value(A, B, domain=(-1.0, 1.0)):
    if A > 0:
        x_uncon = -B / A
        if domain[0] <= x_uncon <= domain[1]:
            return -0.5 * (B * B) / A
    val_lo = 0.5*A*domain[0]**2 + B*domain[0]
    val_hi = 0.5*A*domain[1]**2 + B*domain[1]
    return min(val_lo, val_hi)

# =========================================================
# Sequences
# =========================================================
def make_linear_alternating(n):
    return [{"type": "linear", "g": float((-1.0)**t)} for t in range(1, n+1)]

def make_quadratic_random_centers(n, rng, low=-0.25, high=0.25, q=1.0):
    centers = rng.uniform(low, high, size=n)
    return [{"type": "quadratic", "q": q, "c": float(c)} for c in centers]

# =========================================================
# Algorithms
# =========================================================
def run_ftl(loss_seq, n, domain):
    A = B = 0.0
    x = np.zeros(n + 1)
    for t in range(1, n + 1):
        x[t] = argmin_over_domain(A, B, domain)
        A, B = update_aggregates(A, B, loss_seq[t - 1])
    cum_loss = np.zeros(n + 1); cum_best = np.zeros(n + 1)
    A = B = 0.0
    for t in range(1, n + 1):
        loss_t, _ = eval_loss_and_grad(loss_seq[t - 1], x[t])
        cum_loss[t] = cum_loss[t - 1] + loss_t
        A, B = update_aggregates(A, B, loss_seq[t - 1])
        cum_best[t] = hindsight_min_value(A, B, domain)
    return cum_loss - cum_best

def run_ogd_sqrt_t(loss_seq, n, domain):
    x = np.zeros(n + 1)
    for t in range(1, n):
        _, g = eval_loss_and_grad(loss_seq[t - 1], x[t])
        eta_t = 1.0 / np.sqrt(t)
        x[t + 1] = project_to_domain(x[t] - eta_t * g, domain)
    cum_loss = np.zeros(n + 1); cum_best = np.zeros(n + 1)
    A = B = 0.0
    for t in range(1, n + 1):
        loss_t, _ = eval_loss_and_grad(loss_seq[t - 1], x[t])
        cum_loss[t] = cum_loss[t - 1] + loss_t
        A, B = update_aggregates(A, B, loss_seq[t - 1])
        cum_best[t] = hindsight_min_value(A, B, domain)
    return cum_loss - cum_best

def compute_sigma_eq6(loss_seq, n, domain):
    A = B = 0.0
    a_star = np.zeros(n + 1); Sigma = np.zeros(n + 1)
    for i in range(1, n + 1):
        A, B = update_aggregates(A, B, loss_seq[i - 1])
        a_star[i] = argmin_over_domain(A, B, domain)
        Li_prev = 0.5*A*a_star[i-1]**2 + B*a_star[i-1]
        Li_curr = 0.5*A*a_star[i]**2   + B*a_star[i]
        Sigma[i] = Sigma[i - 1] + (Li_prev - Li_curr)
    return Sigma

def run_smart_deterministic(loss_seq, n, domain):
    Sigma = compute_sigma_eq6(loss_seq, n, domain)
    threshold = 2.0 * np.sqrt(n)
    switch_round = n + 1
    for t in range(1, n + 1):
        if Sigma[t] > threshold:
            switch_round = t + 1
            break
    # FTL trajectory
    A = B = 0.0
    x_ftl = np.zeros(n + 1)
    for t in range(1, n + 1):
        x_ftl[t] = argmin_over_domain(A, B, domain)
        A, B = update_aggregates(A, B, loss_seq[t - 1])
    # OGD trajectory
    x_ogd = np.zeros(n + 1)
    for t in range(1, n):
        _, g = eval_loss_and_grad(loss_seq[t - 1], x_ogd[t])
        eta_t = 1.0 / np.sqrt(t)
        x_ogd[t + 1] = project_to_domain(x_ogd[t] - eta_t * g, domain)
    # SMART trajectory
    x = np.zeros(n + 1)
    for t in range(1, n + 1):
        x[t] = x_ftl[t] if t < switch_round else x_ogd[t]
    cum_loss = np.zeros(n + 1); cum_best = np.zeros(n + 1)
    A = B = 0.0
    for t in range(1, n + 1):
        loss_t, _ = eval_loss_and_grad(loss_seq[t - 1], x[t])
        cum_loss[t] = cum_loss[t - 1] + loss_t
        A, B = update_aggregates(A, B, loss_seq[t - 1])
        cum_best[t] = hindsight_min_value(A, B, domain)
    return cum_loss - cum_best

# =========================================================
# Monte Carlo + plotting
# =========================================================
def summarize(R):
    mean = R.mean(axis=0)
    se = R.std(axis=0, ddof=1) / np.sqrt(R.shape[0])
    return mean, mean - 1.96*se, mean + 1.96*se

def run_trials(loss_seq_maker, n=1000, trials=50, seed=0):
    rng_master = np.random.default_rng(seed)
    domain = (-1.0, 1.0)
    R_ftl, R_ogd, R_smart = np.zeros((trials, n)), np.zeros((trials, n)), np.zeros((trials, n))
    for k in range(trials):
        rng = np.random.default_rng(rng_master.integers(0, 2**31-1))
        losses = loss_seq_maker(n, rng)
        R_ftl[k, :]   = run_ftl(losses, n, domain)[1:]
        R_ogd[k, :]   = run_ogd_sqrt_t(losses, n, domain)[1:]
        R_smart[k, :] = run_smart_deterministic(losses, n, domain)[1:]
    return summarize(R_ftl), summarize(R_ogd), summarize(R_smart)

def plot_and_save(t, stats_ftl, stats_ogd, stats_smart, title, filename):
    (m_ftl, lo_ftl, hi_ftl) = stats_ftl
    (m_ogd, lo_ogd, hi_ogd) = stats_ogd
    (m_smart, lo_smart, hi_smart) = stats_smart
    plt.figure(figsize=(9,6))
    plt.plot(t, m_ftl,   label="FTL", color="C0", linewidth=2)
    plt.fill_between(t, lo_ftl, hi_ftl, alpha=0.2, color="C0")
    plt.plot(t, m_ogd,   label="OGD ($\\eta_t=1/\\sqrt{t}$)", color="C1", linewidth=2)
    plt.fill_between(t, lo_ogd, hi_ogd, alpha=0.2, color="C1")
    plt.plot(t, m_smart, label="SMART", color="C2", linewidth=2)
    plt.fill_between(t, lo_smart, hi_smart, alpha=0.2, color="C2")
    plt.xlabel("n", fontsize=18)
    plt.ylabel("Regret", fontsize=18)
    plt.title(title, fontsize=20)
    plt.legend(fontsize=16)
    plt.xticks(fontsize=14); plt.yticks(fontsize=14)
    plt.tight_layout()
    plt.savefig(filename, bbox_inches="tight")
    plt.close()

# =========================================================
# Run both experiments, save two separate figures
# =========================================================
def run_and_save_all(n=1000, trials=50, seed=0):
    t = np.arange(1, n+1)

    # Linear alternating
    stats_lin = run_trials(lambda n,rng: make_linear_alternating(n), n, trials, seed)
    plot_and_save(
        t, *stats_lin,
        title=r"$\ell_t(a) = g_t\, a,\;\; g_t \in \{\pm 1\}$",
        filename="linear_losses.png"
    )

    # Quadratic random centers
    stats_quad = run_trials(lambda n,rng: make_quadratic_random_centers(n, rng), n, trials, seed)
    plot_and_save(
        t, *stats_quad,
        title=r"Quadratic: $\ell_t(a) = \frac{1}{2}(a-\mu_t)^2$, $\mu_t \sim \mathrm{Unif}[-1/4,1/4]$",
        filename="quadratic_random.png"
    )

# Example run
run_and_save_all(n=1000, trials=50, seed=0)